# alternance-extractor -- benchmark: fine-tuned Qwen2.5-1.5B vs Groq baseline

**Status: draft skeleton, not yet run.** Needs three things that don't exist yet:
1. `data/test/test.jsonl` -- hand-corrected test set (`label/select_test_set.py` then
   `label/review_server.py`), which itself needs the full Groq labelling run finished.
2. `data/test/test_candidates.jsonl` -- already doubles as the Groq baseline's predictions
   file (same run that produced `all_labelled.jsonl` also labelled these 100).
3. A trained LoRA adapter from `notebooks/kaggle_train.ipynb` at `ADAPTER_DIR`.

This notebook: runs the fine-tuned model over the 100 test postings, writes its predictions
in the same shape `label/run_labelling.py` uses, then scores both models with `eval/score.py`
and prints the field-level F1 / exact-match / JSON-validity / latency numbers the README's
claim needs.

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes

In [ ]:
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/mmattar18/alternance-extractor.git"
REPO_ROOT = Path("/kaggle/working/alternance-extractor")

if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)

sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import json
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

from schema.posting import parse_llm_json  # noqa: E402
from label.prompt import SYSTEM_PROMPT, build_messages  # noqa: E402  -- same rules text used to label the data
from eval.score import score_files, print_report, load_test_set, load_predictions, aggregate, aggregate_partial  # noqa: E402

## Config

In [ ]:
BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
# Not "/kaggle/input/alternance-extractor-adapter" -- this Kaggle API version (kagglesdk,
# the OAuth-token CLI) mounts attached datasets at /kaggle/input/datasets/<owner>/<slug>/,
# not the classic /kaggle/input/<slug>/ documented everywhere. Confirmed by walking
# /kaggle/input in a diagnostic run -- the file was genuinely there, just not where every
# tutorial says to look.
ADAPTER_DIR = "/kaggle/input/datasets/mattarmario/alternance-extractor-adapter"

TEST_PATH = REPO_ROOT / "data" / "test" / "test.jsonl"
GROQ_PREDICTIONS_PATH = REPO_ROOT / "data" / "test" / "test_candidates.jsonl"
FINETUNED_PREDICTIONS_PATH = REPO_ROOT / "data" / "test" / "predictions_finetuned.jsonl"
BASE_PREDICTIONS_PATH      = REPO_ROOT / "data" / "test" / "predictions_base.jsonl"

MAX_NEW_TOKENS = 512
MAX_ATTEMPTS = 3  # matches label/groq_client.py's retry-on-invalid-JSON behavior, for a fair comparison

## Load base model + adapter

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

# The un-fine-tuned arm reuses this same object via model.disable_adapter() rather than
# loading a second copy: it guarantees the two arms share byte-identical base weights and
# the same 4-bit quantization, so any measured difference is attributable to the adapter
# alone -- and it avoids holding two 1.5B models on a 14.5GB T4.
print("adapter loaded:", ADAPTER_DIR)

## Inference

Mirrors `label/groq_client.py`'s `extract()`: on invalid JSON, feed the model its own bad
output plus the validation error and ask it to correct itself, up to `MAX_ATTEMPTS` times, so
the JSON-validity-rate comparison against Groq is apples-to-apples rather than giving one side
more chances than the other.

In [ ]:
def extract(raw_text: str, messages_fn) -> dict:
    """messages_fn(raw_text) -> initial message list. Lets the same retry/parse/latency
    logic serve both arms while each gets its intended prompt (see the markdown above)."""
    messages = messages_fn(raw_text)
    start = time.monotonic()
    prompt_tokens = None
    last_error = ""
    for attempt in range(1, MAX_ATTEMPTS + 1):
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        if prompt_tokens is None:
            prompt_tokens = int(inputs["input_ids"].shape[1])  # first-attempt prompt size, for cost analysis
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                temperature=None,
                top_p=None,
                pad_token_id=tokenizer.pad_token_id,
            )
        content = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        posting, error = parse_llm_json(content)
        if posting is not None:
            return {
                "prediction": posting.model_dump(),
                "valid": True,
                "error": None,
                "attempts": attempt,
                "prompt_tokens": prompt_tokens,
                "latency_seconds": time.monotonic() - start,
            }
        last_error = error
        messages.append({"role": "assistant", "content": content})
        messages.append({
            "role": "user",
            "content": f"That was not valid: {error}. Return ONLY a corrected JSON object matching the schema.",
        })
    return {
        "prediction": None,
        "valid": False,
        "error": last_error,
        "attempts": MAX_ATTEMPTS,
        "prompt_tokens": prompt_tokens,
        "latency_seconds": time.monotonic() - start,
    }


def run_arm(messages_fn, out_path, label):
    test_records = [json.loads(l) for l in TEST_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", encoding="utf-8") as f:
        for i, record in enumerate(test_records, 1):
            result = extract(record["raw_text"], messages_fn)
            f.write(json.dumps({"posting_id": record["posting_id"], **result}, ensure_ascii=False) + "\n")
            f.flush()
            if i % 25 == 0 or i == len(test_records):
                print(f"  [{label}] {i}/{len(test_records)}")

In [ ]:
def ft_messages(raw_text):
    # exactly what the model saw in training: system + user, no few-shot
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": raw_text}]

# Arm A: base Qwen2.5-1.5B, NO adapter, few-shot prompted (build_messages) -- the same
# prompt construction Groq gets, i.e. the base model in its best configuration.
print("=== ARM A: base Qwen2.5-1.5B (no adapter, few-shot) ===")
with model.disable_adapter():
    run_arm(build_messages, BASE_PREDICTIONS_PATH, "base")

# Arm B: the fine-tuned adapter, prompted the way it was trained.
print("=== ARM B: fine-tuned Qwen2.5-1.5B (adapter, no few-shot) ===")
run_arm(ft_messages, FINETUNED_PREDICTIONS_PATH, "finetuned")
print("done")

## Score both models

`GROQ_PREDICTIONS_PATH` (`test_candidates.jsonl`) already has Groq's `prediction`/`valid` for
these exact 100 postings from the original labelling run -- no separate Groq re-run needed, and
re-running would just add noise since `temperature=0` already makes it deterministic.

In [ ]:
ARMS = [
    ("GROQ Llama-3.3-70B (few-shot)", GROQ_PREDICTIONS_PATH),
    ("BASE Qwen2.5-1.5B (few-shot, no adapter)", BASE_PREDICTIONS_PATH),
    ("FINE-TUNED Qwen2.5-1.5B (adapter)", FINETUNED_PREDICTIONS_PATH),
]

for label, preds in ARMS:
    print("=" * 25, label, "=" * 25)
    print_report(score_files(TEST_PATH, preds))
    print()

# Strict vs per-item partial credit, side by side. Strict is the headline; partial shows
# where a model is partially right on list fields, which exact set equality hides.
# Reported for every arm, so it cannot flatter one side.
print("=" * 78)
print(f"{'arm':<42}{'strict':>9}{'partial':>9}{'exact':>8}{'fields':>9}")
gold = load_test_set(TEST_PATH)
for label, preds_path in ARMS:
    pr = load_predictions(preds_path)
    recs = [(g, pr[pid][0], pr[pid][1]) for pid, g in gold.items()]
    st, pa = aggregate(recs), aggregate_partial(recs)
    print(f"{label:<42}{st['macro_f1']:>9.3f}{pa['macro_f1_partial']:>9.3f}"
          f"{st['exact_match_rate']:>8.0%}{pa['mean_fields_correct']:>9.1%}")
print("strict/partial = macro F1;  exact = all 14 fields right;  fields = mean fields correct")


## Latency comparison

Read directly off `latency_seconds` in each predictions file -- no cost numbers hardcoded here
deliberately, since Groq's per-token pricing can change and a stale hardcoded figure would be
worse than none. Pull current pricing at analysis time and multiply by measured token counts.

In [ ]:
def stats(path, key="latency_seconds"):
    vals = [json.loads(l)[key] for l in path.read_text(encoding="utf-8").splitlines()
            if l.strip() and json.loads(l).get(key) is not None]
    vals.sort(); n = len(vals)
    return {"n": n, "median": round(vals[n // 2], 2), "mean": round(sum(vals) / n, 2),
            "p95": round(vals[int(n * 0.95)], 2)}

for label, p in [("Groq (rate-limited API)", GROQ_PREDICTIONS_PATH),
                 ("Base Qwen (T4)", BASE_PREDICTIONS_PATH),
                 ("Fine-tuned Qwen (T4)", FINETUNED_PREDICTIONS_PATH)]:
    print(f"{label:<28} latency(s) {stats(p)}")

print()
for label, p in [("Base Qwen", BASE_PREDICTIONS_PATH),
                 ("Fine-tuned Qwen", FINETUNED_PREDICTIONS_PATH)]:
    print(f"{label:<28} prompt_tokens {stats(p, 'prompt_tokens')}")


## Next step

Copy the printed numbers (field-level F1, exact-match rate, JSON-validity rate, latency) into
the README's status section, replacing the "Numbers below are not measured" placeholder --
that's the actual writeup the README says is still pending.